# Module 6.1 — The Cleaning and Transformation Workflow
### Reference & live-demo notebook — AJEBO Finance MFB Home Loan Applications

**Publica Academy Data Analysis Programme · Module 6, Week 7 (Data Storytelling and Visualisation)**

This notebook follows the 17-step cleaning workflow from the Module 6.1 slide deck, in the same
order, on the same dataset. Every code cell here matches a code slide in the deck — run it
alongside the session, or use it as the source of truth when building your own cleaning script.

**Learning outcome:** take a raw dataset through a documented, repeatable cleaning workflow in
pandas, and state the checks that prove the output is trustworthy.

**Dataset:** `ajebo_finance_loan_applications_RAW.csv` — 412 rows, 15 columns. A fictional
Nigerian microfinance bank's home loan applications. No real personal data (Application_ID is a
synthetic reference), so it is safe to explore, chart, and later upload to an AI tool.

> Keep this notebook's outputs — Step 15's validation and Step 16's documentation both depend on
> the exact counts produced by earlier cells.

## Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


---
## Step 1 of 17 — Why data cleaning matters

**The problem:** dirty data doesn't announce itself. A duplicate row, a negative income, a
future-dated application — none of these throw an error. Pandas will happily compute an average
across all of them.

**Why it matters:** someone will act on your number. If AJEBO's credit committee changes Kano
branch policy based on an approval rate that's quietly inflated by duplicate applications, that's
a real business decision built on a mistake nobody can see.

**The mindset shift:** cleaning is the first analytical decision you make, not a chore before the
"real" analysis — and you must be able to defend it as confidently as any chart.

No code in this step — it's the frame for everything that follows.

---
## Step 2 of 17 — Inspecting a raw dataset

**What we're doing:** get oriented before touching a single value — how many rows and columns,
what do the first few rows actually look like, does the shape match what you expected.

**Why it matters:** you cannot plan a cleaning strategy for a dataset you haven't looked at.

**The habit:** make this the first lines you type on any new file, every time, before any
analysis code at all.

In [2]:
# 1. How big is this, and what does it look like?
df = pd.read_csv('ajebo_finance_loan_applications_RAW.csv')
df.shape


(412, 15)

In [3]:
df.head()


  Application_ID Application_Date   Branch_Region Gender Marital_Status Dependents     Education Employment_Type  \
0        AJB2867      02-Jan-2025           Abuja   Male        Married          1      Graduate   Self-Employed   
1        AJB2611       2026-04-03  Port  Harcourt   Male         Single          2  Not Graduate        Salaried   
2        AJB2539       2025-10-03          Ibadan   Male         Single          1      Graduate   Self-Employed   
3        AJB2762       08/06/2025   port harcourt   Male        Married          0      Graduate   Self Employed   
4        AJB2823      10-Jan-2025           Lagos   Male         Single          0  Not Graduate        Salaried   

   Loan_Purpose  Monthly_Income_NGN  Coapplicant_Income_NGN  Loan_Amount_NGN  Loan_Term_Months  Credit_History  \
0  Construction              543900                       0        4311000.0             240.0             0.0   
1    Renovation              288300                       0        2397000.

---
## Step 3 of 17 — Understanding columns and data types

**What we're doing:** check what pandas thinks each column is (its dtype), and compare that
against what the column should be.

**Why it matters:** wrong dtypes fail silently. A term stored as text will sort "120" before "60"
alphabetically, and you won't get an error — you'll get a wrong chart.

**What to look for:** object columns that should be numeric or dates; numeric columns with a
suspiciously low non-null count (a hint of missing values, confirmed properly at Step 4).

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 412 entries, 0 to 411
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Application_ID          412 non-null    str    
 1   Application_Date        412 non-null    str    
 2   Branch_Region           412 non-null    str    
 3   Gender                  398 non-null    str    
 4   Marital_Status          412 non-null    str    
 5   Dependents              400 non-null    str    
 6   Education               412 non-null    str    
 7   Employment_Type         391 non-null    str    
 8   Loan_Purpose            412 non-null    str    
 9   Monthly_Income_NGN      412 non-null    int64  
 10  Coapplicant_Income_NGN  412 non-null    int64  
 11  Loan_Amount_NGN         405 non-null    float64
 12  Loan_Term_Months        404 non-null    float64
 13  Credit_History          376 non-null    float64
 14  Loan_Status             412 non-null    str    
dtype

In [5]:
df.describe()


       Monthly_Income_NGN  Coapplicant_Income_NGN  Loan_Amount_NGN  Loan_Term_Months  Credit_History
count        4.120000e+02              412.000000     4.050000e+02        404.000000      376.000000
mean         2.381420e+05            50457.281553     3.426822e+06        168.742574        0.813830
std          1.260034e+05            54699.219318     2.338392e+06        111.874988        0.389763
min         -4.342000e+05                0.000000     3.040000e+05          0.000000        0.000000
25%          1.722500e+05                0.000000     1.866000e+06         60.000000        1.000000
50%          2.276500e+05            43800.000000     2.919000e+06        180.000000        1.000000
75%          2.895500e+05            93775.000000     4.367000e+06        240.000000        1.000000
max          1.201000e+06           220800.000000     2.000000e+07        360.000000        1.000000

---
## Step 4 of 17 — Detecting missing values

**What problem are we detecting?** blank cells — values never captured. Gender, Dependents,
Employment_Type, Loan_Amount_NGN, Loan_Term_Months and Credit_History all have gaps.

**Why it matters:** missing values silently shrink your sample in some calculations and get
treated as zero in others.

**How we identify it:** `isna().sum()` for counts, `isna().mean()` for the percentage view — 35
missing out of 412 reads very differently as "8.5% of Credit_History is missing."

We are only **detecting** here — deciding what to do about each gap is Step 11.

In [6]:
df.isna().sum()


Application_ID             0
Application_Date           0
Branch_Region              0
Gender                    14
Marital_Status             0
Dependents                12
Education                  0
Employment_Type           21
Loan_Purpose               0
Monthly_Income_NGN         0
Coapplicant_Income_NGN     0
Loan_Amount_NGN            7
Loan_Term_Months           8
Credit_History            36
Loan_Status                0
dtype: int64

In [7]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)


Credit_History            8.7
Employment_Type           5.1
Gender                    3.4
Dependents                2.9
Loan_Term_Months          1.9
Loan_Amount_NGN           1.7
Application_ID            0.0
Education                 0.0
Marital_Status            0.0
Application_Date          0.0
Branch_Region             0.0
Coapplicant_Income_NGN    0.0
Monthly_Income_NGN        0.0
Loan_Purpose              0.0
Loan_Status               0.0
dtype: float64

---
## Step 5 of 17 — Detecting duplicates

**What problem are we detecting?** rows that are exact copies of another row — the same
application counted twice.

**Why it matters:** a duplicated application double-weights that applicant's outcome, nudging the
overall and branch approval rate without representing a real second applicant.

**How we identify it:** `duplicated()` flags rows identical across every column. Because this
dataset has a unique ID, we also check whether any ID appears more than once — a stronger signal
than a full-row match. Both should agree.

In [8]:
df.duplicated().sum(), df['Application_ID'].duplicated().sum()


(np.int64(12), np.int64(12))

In [9]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)
before_rows, after_rows


(412, 400)

**Verify:** every remaining row has a unique ID — `nunique()` equal to row count is proof the
duplicates are gone, not just an assumption that `drop_duplicates()` worked.

In [10]:
assert df['Application_ID'].nunique() == len(df)
print('Verified: no duplicate Application_IDs remain.')


Verified: no duplicate Application_IDs remain.


---
## Step 6 of 17 — Identifying inconsistent categories

**What problem are we detecting?** the same category written multiple ways: "Lagos", "lagos",
"LAGOS " are three different strings to pandas but one branch to everyone else.

**Why it matters:** a `groupby` on Branch_Region would currently split Lagos into several rows
instead of one, understating that branch's true volume and distorting any approval-rate
comparison.

**How we identify it:** `value_counts()` on every categorical column, read in full. More distinct
values than the business has (AJEBO has six branches, not more) is the signal.

In [11]:
df['Branch_Region'].value_counts(dropna=False)


Branch_Region
Lagos             111
Rural North        75
Abuja              63
Ibadan             46
Port Harcourt      45
Kano               32
lagos              10
Abuja               6
port harcourt       4
Lagos               3
abuja               2
LAGOS               2
Port  Harcourt      1
Name: count, dtype: int64

In [12]:
df['Branch_Region'] = (df['Branch_Region']
                        .str.strip()
                        .str.replace(r'\s+', ' ', regex=True)   # collapse internal double spaces
                        .str.title())
df['Branch_Region'].value_counts(dropna=False)


Branch_Region
Lagos            126
Rural North       75
Abuja             71
Port Harcourt     50
Ibadan            46
Kano              32
Name: count, dtype: int64

**Verify:** `Branch_Region` should now show exactly 6 distinct real branches.

In [13]:
df['Branch_Region'].nunique()


6

---
## Step 7 of 17 — Identifying invalid values

**What problem are we detecting?** values that are the right data type but the wrong reality: a
negative income, a zero-month loan term. Nothing about these breaks pandas — they just can't be
true.

**Why it matters:** invalid values are dangerous precisely because they look normal in a quick
scan. A negative income will happily average into your income statistics and quietly drag them
down.

**How we identify it:** encode the business rules as boolean filters.

In [14]:
neg_income = df[df['Monthly_Income_NGN'] < 0]
zero_term = df[df['Loan_Term_Months'] == 0]
bad_credit_flag = df[~df['Credit_History'].isin([0, 1]) & df['Credit_History'].notna()]

len(neg_income), len(zero_term), len(bad_credit_flag)


(2, 1, 0)

We are only **detecting** here — Step 13 decides whether to correct or remove each one.

---
## Step 8 of 17 — Handling incorrect data types

**What problem are we detecting?** `Credit_History` and `Loan_Term_Months` are stored as
`float64` — not because they're naturally decimal, but because missing values force pandas to use
a float so it has room for `NaN`.

**Why it matters:** a float column that's secretly a whole-number or binary category invites
mistakes — averaging a flag like Credit_History (0/1) as if it were continuous produces a number
with no real meaning.

**The trap to avoid:** `Dependents` contains `'3+'` as text. Forcing it numeric with
`astype(int)` will crash. The correct fix isn't conversion — it's recognising the column should
stay categorical.

In [15]:
# Test whether Dependents can safely go numeric
pd.to_numeric(df['Dependents'], errors='coerce').isna().sum(), df['Dependents'].isna().sum()


(np.int64(65), np.int64(12))

The coerce test turns `'3+'` into `NaN` in addition to the genuinely missing values — proof
this column must stay categorical, not become numeric.

In [16]:
df['Dependents'] = df['Dependents'].astype('category')
df['Branch_Region'] = df['Branch_Region'].astype('category')
df[['Dependents', 'Branch_Region']].dtypes


Dependents       category
Branch_Region    category
dtype: object

---
## Step 9 of 17 — Handling dates

**What problem are we detecting?** `Application_Date` mixes three formats in one column: ISO
(2025-06-14), UK-style (14/06/2025), and abbreviated-month (14-Jun-2025). One row is even dated in
the future.

**Why it matters:** as plain text, dates can't be sorted chronologically, filtered by range, or
used to extract a month or year — and a future-dated application slipping through unnoticed
undermines trust in the entire file.

**How we identify and fix it:** convert with `pd.to_datetime()` using mixed-format parsing, then
explicitly check for anything outside a plausible range.

In [17]:
df['Application_Date'] = pd.to_datetime(df['Application_Date'], format='mixed', dayfirst=True)
df['Application_Date'].dtype


dtype('<M8[us]')

In [18]:
future = df[df['Application_Date'] > pd.Timestamp.today()]
len(future), future['Application_ID'].tolist()


(8, ['AJB2730', 'AJB2787', 'AJB2582', 'AJB2805', 'AJB2558', 'AJB2723', 'AJB2630', 'AJB2760'])

**Verify:** the column is now a real datetime, and the future-date row is on our list to
resolve at Step 13 (it cannot be safely corrected, only removed).

---
## Step 10 of 17 — Standardising text

**What problem are we detecting?** values needing genuine re-mapping, not just tidying — a "Y" /
"N" shorthand for Married/Single, or "Self Employed" sitting next to "Self-Employed". Step 6
already handled simple case/whitespace issues; this step goes further.

**Why it matters:** one mapping dictionary applied consistently beats fixing each column by hand
— and it's the reusable piece that feeds Step 17's repeatable pipeline. Notice the function below
also collapses internal double spaces with a small regex — the same fix Step 6 needed for
"Port  Harcourt", generalised so every text column gets it automatically from now on.

**How we fix it:** one small function — strip, standardise case, then apply an explicit mapping
dictionary for anything that needs real re-labelling.

In [19]:
df['Gender'].value_counts(dropna=False)


Gender
Male       235
Female     111
NaN         14
male        13
 Male        9
Female       7
FEMALE       6
MALE         4
female       1
Name: count, dtype: int64

In [20]:
marital_map = {'Y': 'Married', 'N': 'Single'}
employ_map = {'Self Employed': 'Self-Employed'}

def clean_text(series, mapping=None):
    s = series.str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
    return s.replace(mapping) if mapping else s

df['Gender'] = clean_text(df['Gender'])
df['Marital_Status'] = clean_text(df['Marital_Status'], marital_map)
df['Employment_Type'] = clean_text(df['Employment_Type'], employ_map)

df['Gender'].value_counts(dropna=False)


Gender
Male      261
Female    125
NaN        14
Name: count, dtype: int64

In [21]:
df['Marital_Status'].value_counts(dropna=False), df['Employment_Type'].value_counts(dropna=False)


(Marital_Status
Married    246
Single     154
Name: count, dtype: int64, Employment_Type
Salaried         255
Self-Employed    125
NaN               20
Name: count, dtype: int64)

**Verify:** `value_counts()` on every cleaned text column shows only its documented valid
values, nothing left over.

---
## Step 11 of 17 — Handling missing values appropriately

**What problem are we solving?** Step 4 counted the gaps. Now we decide what to do about each —
and the honest answer is: it depends on the column, not one blanket rule for the whole dataset.

**Why it matters:** `Credit_History` is the strongest predictor of approval in this dataset.
Imputing its missing values with the most common value would quietly manufacture a pattern that
isn't real data. For a low-stakes column like Gender, the same shortcut is much lower risk.

**The decision framework:** could you drop the rows without meaningfully shrinking or biasing the
dataset? If not, is there a defensible fill value? If neither, keep the gap visible as its own
category and document why.

In [22]:
# Low-stakes categorical: fill with an explicit label
df['Gender'] = df['Gender'].fillna('Not Stated')

# Numeric, roughly symmetric: fill with the median
df['Loan_Amount_NGN'] = df['Loan_Amount_NGN'].fillna(df['Loan_Amount_NGN'].median())
df['Loan_Term_Months'] = df['Loan_Term_Months'].fillna(df['Loan_Term_Months'].median())

# Low-stakes categorical: fill with an explicit label
df['Employment_Type'] = df['Employment_Type'].fillna('Not Stated')
df['Dependents'] = df['Dependents'].astype(object).fillna('Not Stated').astype('category')

# High-stakes driver of the outcome: drop, don't guess
rows_before_credit_drop = len(df)
df = df.dropna(subset=['Credit_History'])
rows_dropped_for_credit_history = rows_before_credit_drop - len(df)

df.isna().sum()


Application_ID            0
Application_Date          0
Branch_Region             0
Gender                    0
Marital_Status            0
Dependents                0
Education                 0
Employment_Type           0
Loan_Purpose              0
Monthly_Income_NGN        0
Coapplicant_Income_NGN    0
Loan_Amount_NGN           0
Loan_Term_Months          0
Credit_History            0
Loan_Status               0
dtype: int64

**Verify:** `isna().sum()` returns zero everywhere we intended to resolve, and we can name the
reason behind each column's chosen strategy (see the markdown above).

---
## Step 12 of 17 — Detecting potential outliers

**What problem are we detecting?** values that are technically valid but sit far outside the
normal range — a self-employed applicant earning several times the typical income. Unlike Step
7's invalid values, these could genuinely be real.

**Why it matters:** treat every outlier as an error and you'll delete real, important customers.
AJEBO's highest earners are often self-employed business owners — exactly the applicants a branch
manager most wants visibility on.

**How we identify it:** the IQR method (values beyond 1.5× the interquartile range) gives a
repeatable, defensible rule rather than an eyeballed cutoff.

In [23]:
q1, q3 = df['Monthly_Income_NGN'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr

outliers = df[df['Monthly_Income_NGN'] > upper]
len(outliers)


11

In [24]:
outliers['Employment_Type'].value_counts()


Employment_Type
Self-Employed    11
Name: count, dtype: int64

Most flagged rows are Self-Employed — a business reason the value is plausible, not a coding
error. We keep these rows; the decision and its reasoning belong in the Step 16 cleaning log.

---
## Step 13 of 17 — Removing or correcting errors

**What we're deciding now:** Steps 7 and 9 detected three concrete errors: negative incomes, a
zero-month term, and one future-dated application. Now we act on each, individually.

**Correct vs. remove:** correct only when you have solid grounds to know the true value — a
negative income is very likely a sign error, so taking the absolute value is defensible. A
zero-month term or a wrong-year date can't be safely guessed, so those rows are dropped.

In [25]:
# Negative income: near-certainly a sign error -> correct
df['Monthly_Income_NGN'] = df['Monthly_Income_NGN'].abs()

# Zero-month term: cannot infer a true value -> remove
rows_before = len(df)
df = df[df['Loan_Term_Months'] != 0]
removed_zero_term = rows_before - len(df)

# Future date: cannot infer a true date -> remove
rows_before = len(df)
df = df[df['Application_Date'] <= pd.Timestamp.today()]
removed_future_date = rows_before - len(df)

removed_zero_term, removed_future_date, len(df)


(1, 8, 356)

**Verify:** re-running the exact Step 7 checks now returns zero rows for every one of them.

In [26]:
assert (df['Monthly_Income_NGN'] >= 0).all()
assert (df['Loan_Term_Months'] != 0).all()
assert df['Application_Date'].max() <= pd.Timestamp.today()
print('Verified: no negative incomes, no zero-month terms, no future dates remain.')


Verified: no negative incomes, no zero-month terms, no future dates remain.


---
## Worked example — one application, before and after

Unlike the illustrative composite on the slide deck's "Before → Cleaning → After" slide, this is a
**real row** from the raw file, chosen because it genuinely has two of today's problems at once:
a blank `Employment_Type` and a non-ISO `Application_Date`.

**Before cleaning**, `AJB2548` looked like this: `Employment_Type` was blank, and
`Application_Date` was the text string `"15-Jan-2026"`.

Run the cell below to see it in its current, fully cleaned state.

In [27]:
df.loc[df['Application_ID'] == 'AJB2548',
       ['Application_ID', 'Branch_Region', 'Employment_Type', 'Application_Date',
        'Monthly_Income_NGN', 'Credit_History']]


   Application_ID Branch_Region Employment_Type Application_Date  Monthly_Income_NGN  Credit_History
63        AJB2548         Lagos      Not Stated       2026-01-15              128100             1.0

---
## Cleaned does not automatically mean trustworthy

You've fixed every problem you found. That is not the same as proving nothing is left. A cleaning
script can run without a single error and still hand you a dataset that quietly lies — a wrong
`astype()` that silently truncates values, an imputation that erases a real pattern,
`drop_duplicates()` that kept the wrong copy of a pair. "It ran" is not a check. "It looks fine"
is not a check.

Trustworthy data needs deliberate checks — which is exactly what Steps 15 and 16 give us below.

---
## Step 14 of 17 — Creating useful derived columns

**What we're doing:** building new columns from the cleaned ones that answer questions the raw
data can't answer directly.

**Why it matters:** a derived column done once, correctly, on clean inputs saves every downstream
analyst from recalculating it slightly differently in Module 6.2's charts.

**The rule:** derived columns are built only after cleaning, never before — a ratio calculated
from a still-dirty income column just launders the dirt into a brand new column.

In [28]:
df['Total_Household_Income_NGN'] = df['Monthly_Income_NGN'] + df['Coapplicant_Income_NGN']
df['Loan_To_Income_Ratio'] = (df['Loan_Amount_NGN'] / df['Total_Household_Income_NGN']).round(2)
df['Application_Year'] = df['Application_Date'].dt.year

df[['Total_Household_Income_NGN', 'Loan_To_Income_Ratio', 'Application_Year']].head()


   Total_Household_Income_NGN  Loan_To_Income_Ratio  Application_Year
0                      543900                  7.93              2025
1                      288300                  8.31              2026
2                      283900                  9.44              2025
3                      273600                 15.10              2025
4                      233900                  3.77              2025

---
## Step 15 of 17 — Validating the cleaned dataset

**What we're doing:** re-running every detection check from Steps 4 through 9, on purpose,
against the finished dataset — and confirming each one now comes back clean.

### Data-quality checklist
- **Completeness** — are the missing values we decided to resolve actually resolved?
- **Accuracy** — do values reflect reality?
- **Consistency** — does the same category always look the same way?
- **Validity** — does every value obey its business rule?
- **Uniqueness** — is every application counted exactly once?
- **Correct data types** — does `df.dtypes` match what each column represents?
- **Plausible ranges** — do min/max and the outlier check look like a real lending business?

In [29]:
assert df.isna().sum().sum() == 0, "Unexpected missing values remain"
assert df['Application_ID'].nunique() == len(df), "Duplicate IDs remain"
assert (df['Monthly_Income_NGN'] >= 0).all(), "Negative income remains"
assert (df['Loan_Term_Months'] > 0).all(), "Zero-month term remains"
assert df['Application_Date'].max() <= pd.Timestamp.today(), "Future date remains"
assert df['Branch_Region'].nunique() == 6, "Region category count is off"
assert set(df['Gender'].unique()) <= {'Male', 'Female', 'Not Stated'}, "Unexpected Gender value"

print('All validation checks passed.')
print(f'Final shape: {df.shape}')


All validation checks passed.
Final shape: (356, 18)


---
## Step 16 of 17 — Documenting the cleaning process

**What we're doing:** writing down, in plain language, every decision made and why. Good
documentation includes starting/ending row counts, each column's missing-value strategy, and
every corrected or removed value with its reason.

The cell below assembles a cleaning log automatically from the counters we tracked in earlier
cells — this is the same discipline as Step 17's repeatable pipeline, applied to documentation.

In [30]:
cleaning_log = f'''
AJEBO Finance loan applications — cleaning log
Source file: ajebo_finance_loan_applications_RAW.csv (412 rows)

1. Removed 12 exact duplicate rows. Result: 400 unique applications.
2. Standardised Branch_Region, Gender, Employment_Type, Marital_Status
   casing and whitespace; mapped 'Y'/'N' to Married/Single.
3. Parsed Application_Date from three mixed formats to a single datetime
   column. Removed {removed_future_date} row(s) dated in the future (unverifiable).
4. Missing values: Gender, Employment_Type, Dependents -> 'Not Stated'.
   Loan_Amount_NGN, Loan_Term_Months -> median. Credit_History
   ({rows_dropped_for_credit_history} rows) -> dropped, given its role as the
   strongest driver of Loan_Status.
5. Corrected negative Monthly_Income_NGN values via abs().
   Removed {removed_zero_term} row(s) with Loan_Term_Months == 0 (unverifiable).
6. Added Total_Household_Income_NGN, Loan_To_Income_Ratio, Application_Year
   as derived columns.
7. All data-quality checklist items confirmed. Final shape: {df.shape}.
'''
print(cleaning_log)



AJEBO Finance loan applications — cleaning log
Source file: ajebo_finance_loan_applications_RAW.csv (412 rows)

1. Removed 12 exact duplicate rows. Result: 400 unique applications.
2. Standardised Branch_Region, Gender, Employment_Type, Marital_Status
   casing and whitespace; mapped 'Y'/'N' to Married/Single.
3. Parsed Application_Date from three mixed formats to a single datetime
   column. Removed 8 row(s) dated in the future (unverifiable).
4. Missing values: Gender, Employment_Type, Dependents -> 'Not Stated'.
   Loan_Amount_NGN, Loan_Term_Months -> median. Credit_History
   (35 rows) -> dropped, given its role as the
   strongest driver of Loan_Status.
5. Corrected negative Monthly_Income_NGN values via abs().
   Removed 1 row(s) with Loan_Term_Months == 0 (unverifiable).
6. Added Total_Household_Income_NGN, Loan_To_Income_Ratio, Application_Year
   as derived columns.
7. All data-quality checklist items confirmed. Final shape: (356, 18).



---
## Step 17 of 17 — Making the workflow repeatable

**What we're doing:** wrapping the entire sequence into a single function that takes a raw file
path and returns a validated, clean dataset.

**Why it matters:** next month's AJEBO export will have the same kinds of problems in different
rows. A repeatable pipeline means you clean it in seconds, with the exact same standards, instead
of re-deciding everything from scratch.

Run the cell below, then the one after it, to confirm the whole pipeline reproduces this
notebook's result from the untouched raw file.

In [31]:
def clean_ajebo_data(filepath):
    """
    Clean a raw AJEBO Finance loan-applications export end to end.

    Assumes the input file has the same 15 columns as
    ajebo_finance_loan_applications_RAW.csv. Returns a validated,
    documented DataFrame. Raises an AssertionError if any validation
    check fails, rather than returning silently-wrong data.
    """
    data = pd.read_csv(filepath)

    # Step 5 — duplicates
    data = data.drop_duplicates()

    # Steps 6 & 10 — standardise text
    marital_map = {'Y': 'Married', 'N': 'Single'}
    employ_map = {'Self Employed': 'Self-Employed'}

    def _clean_text(series, mapping=None):
        s = series.str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
        return s.replace(mapping) if mapping else s

    data['Branch_Region'] = _clean_text(data['Branch_Region'])
    data['Gender'] = _clean_text(data['Gender'])
    data['Marital_Status'] = _clean_text(data['Marital_Status'], marital_map)
    data['Employment_Type'] = _clean_text(data['Employment_Type'], employ_map)

    # Step 8 — dtypes
    data['Dependents'] = data['Dependents'].astype('category')
    data['Branch_Region'] = data['Branch_Region'].astype('category')

    # Step 9 — dates
    data['Application_Date'] = pd.to_datetime(data['Application_Date'], format='mixed', dayfirst=True)

    # Step 11 — missing values, column by column
    data['Gender'] = data['Gender'].fillna('Not Stated')
    data['Employment_Type'] = data['Employment_Type'].fillna('Not Stated')
    data['Dependents'] = data['Dependents'].astype(object).fillna('Not Stated').astype('category')
    data['Loan_Amount_NGN'] = data['Loan_Amount_NGN'].fillna(data['Loan_Amount_NGN'].median())
    data['Loan_Term_Months'] = data['Loan_Term_Months'].fillna(data['Loan_Term_Months'].median())
    data = data.dropna(subset=['Credit_History'])

    # Step 13 — correct or remove errors
    data['Monthly_Income_NGN'] = data['Monthly_Income_NGN'].abs()
    data = data[data['Loan_Term_Months'] != 0]
    data = data[data['Application_Date'] <= pd.Timestamp.today()]

    # Step 14 — derived columns
    data['Total_Household_Income_NGN'] = data['Monthly_Income_NGN'] + data['Coapplicant_Income_NGN']
    data['Loan_To_Income_Ratio'] = (data['Loan_Amount_NGN'] / data['Total_Household_Income_NGN']).round(2)
    data['Application_Year'] = data['Application_Date'].dt.year

    # Step 15 — validate before returning
    assert data.isna().sum().sum() == 0
    assert data['Application_ID'].nunique() == len(data)
    assert (data['Monthly_Income_NGN'] >= 0).all()
    assert (data['Loan_Term_Months'] > 0).all()
    assert data['Application_Date'].max() <= pd.Timestamp.today()
    assert data['Branch_Region'].nunique() == 6

    return data


In [32]:
result_1 = clean_ajebo_data('ajebo_finance_loan_applications_RAW.csv')
result_2 = clean_ajebo_data('ajebo_finance_loan_applications_RAW.csv')

result_1.shape, result_1.equals(result_2)


((356, 18), True)

**Verify:** running `clean_ajebo_data()` twice on the same raw file produces byte-identical
output both times — true repeatability, not just "it worked once." 

---
## Recap

- **Detect** — missing values, duplicates, inconsistent categories, invalid values, wrong
  dtypes, outliers.
- **Fix** — standardise text, parse dates, handle missing values by column, correct or remove
  errors deliberately.
- **Prove** — build derived columns on clean inputs, validate with assertions, document every
  decision, make it repeatable.

## Next: Module 6.2 — Why clean data matters for visualisation

Save your cleaned `df` (or the output of `clean_ajebo_data()`) — you'll chart it directly in the
next session.

In [33]:
df.to_csv('ajebo_finance_loan_applications_CLEANED.csv', index=False)
print('Saved: ajebo_finance_loan_applications_CLEANED.csv', df.shape)


Saved: ajebo_finance_loan_applications_CLEANED.csv (356, 18)
